In [3]:
import pandas as pd
import numpy as np

ratings = pd.read_csv("../data/raw/ratings.csv")

ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26024289 entries, 0 to 26024288
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  int64  
dtypes: float64(1), int64(3)
memory usage: 794.2 MB


In [3]:
print(ratings["movieId"].sample(10, random_state=42).tolist())

[163, 55118, 3499, 2571, 4973, 56286, 3360, 3755, 5577, 4343]


In [ ]:
# movies_meta = pd.read_csv(
#     "../data/raw/movies_metadata.csv",
#     low_memory=False
# )

# # Remove corrupted rows
# movies_meta = movies_meta[movies_meta['id'].str.isdigit()]
# movies_meta['id'] = movies_meta['id'].astype(int)

In [ ]:
# movies_meta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 45463 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45463 non-null  object 
 1   belongs_to_collection  4491 non-null   object 
 2   budget                 45463 non-null  object 
 3   genres                 45463 non-null  object 
 4   homepage               7779 non-null   object 
 5   id                     45463 non-null  int32  
 6   imdb_id                45446 non-null  object 
 7   original_language      45452 non-null  object 
 8   original_title         45463 non-null  object 
 9   overview               44509 non-null  object 
 10  popularity             45460 non-null  object 
 11  poster_path            45077 non-null  object 
 12  production_companies   45460 non-null  object 
 13  production_countries   45460 non-null  object 
 14  release_date           45376 non-null  object 
 15  revenue

In [8]:
from pandas.core.common import random_state
print(movies["id"].sample(10, random_state=42).tolist())

['411405', '42492', '12143', '9976', '46761', '268725', '62297', '184885', '156145', '306745']


In [ ]:
links = pd.read_csv('../data/raw/links.csv',
    dtype={'movieId': 'int64', 'tmdbId': 'float64'}
)

links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45843 entries, 0 to 45842
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  45843 non-null  int64  
 1   imdbId   45843 non-null  int64  
 2   tmdbId   45624 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 1.0 MB


In [17]:
# Validate Overlap before merge
ratings_unique = ratings["movieId"].nunique()
links_unique = ratings["movieId"].nunique()

common_id = set(ratings["movieId"]) & set(links["movieId"])

print(f"\nUnique ratings movieIds: {ratings_unique}")
print(f"\nUnique links movieIds: {links_unique}")
print(f"\nOverlap %: {100* len(common_id) / ratings_unique:.4f}%")
print(f"\nOverlap %: {100* len(common_id) / links_unique:.4f}%")


Unique ratings movieIds: 45115

Unique links movieIds: 45115

Overlap %: 100.0000%

Overlap %: 100.0000%


In [19]:
# Merge ratings to links
ratings_with_tmdb = ratings.merge(
    links[["movieId", "tmdbId"]],
    on="movieId",
    how="left")

missing_ratio = ratings_with_tmdb["tmdbId"].isna().mean()
print(f"\nMissing tmdbId ratio after merge: {missing_ratio}")

# Decision Gate
if missing_ratio > 0.01:
    print("Warning: More than 1% ratings missing tmdbId. Investigate.")
else:
    print("Mapping integrity accepted")

# Drop unmatch
ratings_clean = ratings_with_tmdb.dropna(subset=["tmdbId"]).copy()
ratings_clean["tmdbId"] = ratings_clean["tmdbId"].astype(int)

print(f"\nOriginal rows: {len(ratings):,}")
print(f"Retained rows: {len(ratings_clean):,}")
print(f"Retention: {100* len(ratings_clean) / len(ratings):.2f}%")


#  Saved processed file
ratings_clean.to_csv("../data/processed/ratings_clean.csv", index=False)
print("Saved: data/processed/ratings_clean.csv")


Missing tmdbId ratio after merge: 0.0005188614374824996
Mapping integrity accepted

Original rows: 26,024,289
Retained rows: 26,010,786
Retention: 99.95%
Saved: data/processed/ratings_clean.csv


In [17]:
movies = pd.read_csv("../data/raw/movies_metadata.csv", low_memory=False)

In [18]:
movies.head(1)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0


In [19]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [20]:
import ast
print("Initial shape:", movies.shape)

movies = movies[movies["id"].apply(lambda x: str(x).isdigit())].copy()
movies["id"] = movies["id"].astype(int)
print("After numeric ID format", movies.shape)

Initial shape: (45466, 24)
After numeric ID format (45463, 24)


In [21]:
# Adult Filter
movies = movies[movies["adult"] == "False"].copy()
print("After adult filter:", movies.shape)

After adult filter: (45454, 24)


In [22]:
# Deduplicate by ID
movies = movies.drop_duplicates(subset=["id"], keep="first")
assert movies["id"].is_unique
print("After duplication:", movies.shape)

After duplication: (45424, 24)


In [23]:
numeric_cols = ['budget', 'revenue', 'popularity', 'vote_count', 'vote_average', 'runtime']

for col in numeric_cols:
    if col in movies.columns:
        movies[col] = pd.to_numeric(movies[col], errors='coerce')

# Replace zero budget/revenue with NaN 
movies['budget'] = movies['budget'].replace(0, np.nan)
movies['revenue'] = movies['revenue'].replace(0, np.nan)

# Release Date Parsing 
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['release_year'] = movies['release_date'].dt.year.astype('Int64')

# Parse Genres 
def parse_list_field(x, key='name'):
    if pd.isna(x):
        return []
    try:
        parsed = ast.literal_eval(x)
        return [item[key] for item in parsed if key in item]
    except:
        return []

movies['genres_list'] = movies['genres'].apply(lambda x: parse_list_field(x, 'name'))
movies['primary_genre'] = movies['genres_list'].apply(
    lambda x: x[0] if len(x) > 0 else None
)

# Parse production companies 
movies['production_companies_list'] = movies['production_companies'].apply(
    lambda x: parse_list_field(x, 'name')
)

print("\nMissing primary genre:", movies['primary_genre'].isna().mean() * 100, "%")

# Runtime Outlier Inspection
print("\n=== RUNTIME OUTLIER INSPECTION ===")
runtime_outliers = movies[movies['runtime'] > 500]
print(f"Movies with runtime > 500 min: {len(runtime_outliers)}")

if len(runtime_outliers) > 0:
    print("\nSample of extreme runtime movies (for inspection):")
    print(runtime_outliers[['title', 'runtime', 'genres_list', 'release_year']].head(10))
    
    # Check for negative or zero runtime
    invalid_runtime = movies[movies['runtime'] <= 0]
    print(f"\nMovies with runtime <= 0: {len(invalid_runtime)}")

# Vote average statistics
print("\n=== VOTE AVERAGE STATISTICS ===")
print(movies['vote_average'].describe())
print(f"Movies with zero votes: {(movies['vote_count'] == 0).sum()}")

# Final Verification
print("\n=== FINAL VERIFICATION ===")
print("Final shape:", movies.shape)
print("ID unique?", movies['id'].is_unique)
print("Columns with missing data:")
print(movies.isnull().sum().sort_values(ascending=False).head(10))


Missing primary genre: 5.376012680521311 %

=== RUNTIME OUTLIER INSPECTION ===
Movies with runtime > 500 min: 30

Sample of extreme runtime movies (for inspection):
                                title  runtime                  genres_list  \
6747                            Shoah    566.0                [Documentary]   
9117       From the Earth to the Moon    720.0         [Documentary, Drama]   
12922                      John Adams    501.0             [History, Drama]   
13051                   Into the West    552.0  [Adventure, Drama, Western]   
13767           Berlin Alexanderplatz    931.0                      [Drama]   
13953  Heimat: A Chronicle of Germany    925.0             [Drama, History]   
18629                   The Civil War    680.0                [Documentary]   
19158                         The War    874.0  [Documentary, History, War]   
19965                            Jazz   1140.0                [Documentary]   
20963    New York: A Documentary Film    600

In [24]:
zero_runtime = movies[movies["runtime"]<=0]
print(zero_runtime["runtime"].value_counts(dropna=False))
print(zero_runtime[["title", "runtime"]].head(10))

runtime
0.0    1556
Name: count, dtype: int64
                                  title  runtime
222                           Dream Man      0.0
224          Destiny Turns on the Radio      0.0
398                        Dos Crímenes      0.0
554           The Beans of Egypt, Maine      0.0
667              The Run of the Country      0.0
679                Under The Domim Tree      0.0
685                To Cross the Rubicon      0.0
776  The Old Lady Who Walked in the Sea      0.0
792                   A Boy Called Hate      0.0
832                     Mille bolle blu      0.0


In [36]:
# Check ID range and type
print(f"ID type: {movies['id'].dtype}")
print(f"ID range: {movies['id'].min()} to {movies['id'].max()}")
print(f"Unique IDs: {movies['id'].nunique()} (matches shape: {len(movies)})")

# Check for any remaining adult content
if "adult" in movies.columns:
    print(f"Adult values remaining: {movies['adult'].unique()}")

# Verify budget/revenue are properly NaN (not 0)
budget_zeros = (movies["budget"] == 0).sum()
revenue_zeros = (movies["revenue"] == 0).sum()
print(f"Budget still zero: {budget_zeros}")
print(f"Revenue still zero: {revenue_zeros}")

if budget_zeros == 0 and revenue_zeros == 0:
    print("Budget/revenue properly converted to NaN")

# Check runtime <=0 are actually NaN(not negative numbers)
runtime_issues = movies[movies['runtime'] < 0].shape[0]
print(f"Negative runtime values: {runtime_issues}")
if runtime_issues == 0:
    print("No negative runtimes")

# Verify genres_list is properly parsed
sample_genres = movies["genres_list"].iloc[0]
print(f"\nSample genres (first movie): {sample_genres}")
print(f"Type: {type(sample_genres)}")

ID type: int32
ID range: 2 to 469172
Unique IDs: 45424 (matches shape: 45424)
Adult values remaining: ['False']
Budget still zero: 0
Revenue still zero: 0
Budget/revenue properly converted to NaN
Negative runtime values: 0
No negative runtimes

Sample genres (first movie): ['Animation', 'Comedy', 'Family']
Type: <class 'list'>


In [74]:
# Load and inspect credits.csv

credits = pd.read_csv('../data/raw/credits.csv')

print("=" * 40)
print("CREDITS.CSV - INITIAL INSPECTION")
print("=" * 40)

print(f"\nShape: {credits.shape}")
print(f"\nColumns: {credits.columns.tolist()}")
print(f"\nData types:\n{credits.dtypes}")
print(f"\nFirst row cast (sample):")
print(credits['cast'].iloc[0][:200], "...") # First 200 char
print(f"\nFirst row crew (sample):")
print(credits['crew'].iloc[0][:200], "...")
print(f"\nUnique movie IDs in credits: {credits['id'].nunique()}")
print(f"Total rows: {len(credits)}")
print(f"Rows per movie (if any duplicates): {len(credits)/credits['id'].nunique():.2f}")

CREDITS.CSV - INITIAL INSPECTION

Shape: (45476, 3)

Columns: ['cast', 'crew', 'id']

Data types:
cast    object
crew    object
id       int64
dtype: object

First row cast (sample):
[{'cast_id': 14, 'character': 'Woody (voice)', 'credit_id': '52fe4284c3a36847f8024f95', 'gender': 2, 'id': 31, 'name': 'Tom Hanks', 'order': 0, 'profile_path': '/pQFoyx7rp09CJTAb932F2g8Nlho.jpg'}, {'c ...

First row crew (sample):
[{'credit_id': '52fe4284c3a36847f8024f49', 'department': 'Directing', 'gender': 2, 'id': 7879, 'job': 'Director', 'name': 'John Lasseter', 'profile_path': '/7EdqiNbr4FRjIhKHyPPdFfEEEFG.jpg'}, {'credit ...

Unique movie IDs in credits: 45432
Total rows: 45476
Rows per movie (if any duplicates): 1.00


### Interpretation:

#### Perfect three columns, with cast and crew as JSON strings, id as integer TMDB ID.
- Shape: (45476, 3)
- Columns: ['cast', 'crew', 'id']
- Data types: all object except id (int64)

#
- 45,476 total rows

- 45,432 unique movie IDs

- 44 duplicate IDs (45,476 - 45,432)

What this means: 44 movies have multiple credits entries. This is unusual but possible if:

The same movie appears twice in the credits file (data error)

The movie has multiple credits records (unlikely—credits should be one row per movie)

#### Investigate duplicate IDs:

In [75]:
# Find duplicate IDs in credits
duplicate_credits = credits[credits['id'].duplicated(keep=False)]
print(f"Number of duplicate IDs entries: {len(duplicate_credits)}")
print(f"Unique duplicate IDs: {duplicate_credits['id'].nunique()}")
print(f"\nSample of duplicate IDs:")
print(duplicate_credits.groupby('id').size().sort_values(ascending=False).head(10))

# Check one Example
example_dup_id = duplicate_credits['id'].iloc[0]
print(f"\nExaminig ID {example_dup_id}")
dup_rows = duplicate_credits[duplicate_credits['id'] == example_dup_id]
for idx, row in dup_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"    cast length: {len(row['cast'])}")
    print(f"    crew length: {len(row['crew'])}")

Number of duplicate IDs entries: 87
Unique duplicate IDs: 43

Sample of duplicate IDs:
id
141971    3
132641    2
99080     2
105045    2
109962    2
110428    2
116723    2
119916    2
123634    2
125458    2
dtype: int64

Examinig ID 105045

Row 676:
    cast length: 1776
    crew length: 719

Row 1465:
    cast length: 1776
    crew length: 719


#### Check if the actual content is identical

In [76]:
id_105045_rows = credits[credits['id'] == 105045]
print("Are cast strings identical?", id_105045_rows.iloc[0]['cast'] == id_105045_rows.iloc[1]['cast'])
print("Are crew strings identical?", id_105045_rows.iloc[0]['crew'] == id_105045_rows.iloc[1]['crew'])

Are cast strings identical? True
Are crew strings identical? True


#### Find the movie without credits

In [77]:
movies_without_credits = movies[~movies['id'].isin(credits['id'])]
print(f"Move missing credits:")
print(movies_without_credits[['id', 'title', 'release_year']])

Move missing credits:
           id         title  release_year
42883  401840  School's out          2017


In [78]:
# Quick check of the missing movie
missing_movie = movies[movies['id'] == 401840]
print(missing_movie[['title', 'release_year', 'genres_list']].to_dict())

{'title': {42883: "School's out"}, 'release_year': {42883: 2017}, 'genres_list': {42883: []}}


#### Deduplicate credits before parsing

In [79]:
credits = credits.drop_duplicates(subset=['id'], keep='first').copy()
print(f"Credits after duplication: {credits.shape}")
print(f"Unique IDs: {credits['id'].nunique()}")
assert credits['id'].is_unique, "Still has duplicates"

Credits after duplication: (45432, 3)
Unique IDs: 45432


In [80]:
def parse_cast(cast_str, max_actors=5):
    """Extract top N actor names from cast JSON"""
    try:
        cast_list = ast.literal_eval(cast_str)
        # Sort by order and take top max_actors
        sorted_cast = sorted(cast_list, key=lambda x: x.get("order", 999))[:max_actors]
        return [actor['name'] for actor in sorted_cast if 'name' in actor]
    except:
        return []

def parse_crew_by_job(crew_str, target_jobs):
    """Extract crew names for specific jobs (Director, Writer, Screenplay)"""
    try:
        crew_list = ast.literal_eval(crew_str)
        #Filter by job title
        matched = [member['name'] for member in crew_list
        if member.get('job') in target_jobs]
        
        return list(set(matched))
    except:
        return []

#Applying Parsing
credits["cast_top5"] = credits['cast'].apply(parse_cast) 
credits["directors"] = credits['crew'].apply(lambda x: parse_crew_by_job(x, ["Director"]))
credits["writers"] = credits['crew'].apply(lambda x: parse_crew_by_job(x, ['Screenplay', "Writer"]))

#Verify 
print("\nSample parsed data")
print(credits[['id', 'cast_top5', 'directors', 'writers']].head(3))



Sample parsed data
      id                                          cast_top5        directors  \
0    862  [Tom Hanks, Tim Allen, Don Rickles, Jim Varney...  [John Lasseter]   
1   8844  [Robin Williams, Jonathan Hyde, Kirsten Dunst,...   [Joe Johnston]   
2  15602  [Walter Matthau, Jack Lemmon, Ann-Margret, Sop...  [Howard Deutch]   

                                             writers  
0  [Joss Whedon, Alec Sokolow, Joel Cohen, Andrew...  
1      [Jim Strain, Jonathan Hensleigh, Greg Taylor]  
2                              [Mark Steven Johnson]  


In [82]:
# Check overlap between movies and credits
movies_ids = set(movies['id'])
credits_ids = set(credits['id'])

print("=== MOVIES vs CREDITS ALIGNMENT ===")
print(f"Movies: {len(movies_ids)} unique IDs")
print(f"Credits: {len(credits_ids)} unique IDs")

# Movies that exist in credits
movies_with_credits = len(movies_ids.intersection(credits_ids))
print(f"Movies with credits: {movies_with_credits} ({movies_with_credits/len(movies_ids)*100:.2f}%)")

# Credits that don't exist in movies
credits_orphaned = len(credits_ids - movies_ids)
print(f"Credits for non-existent movies: {credits_orphaned}")

# Movies missing credits
movies_missing_credits = len(movies_ids - credits_ids)
print(f"Movies missing credits: {movies_missing_credits} ({movies_missing_credits/len(movies_ids)*100:.2f}%)")

=== MOVIES vs CREDITS ALIGNMENT ===
Movies: 45424 unique IDs
Credits: 45432 unique IDs
Movies with credits: 45423 (100.00%)
Credits for non-existent movies: 9
Movies missing credits: 1 (0.00%)


#### Merge movies with credits (LEFT JOIN - keep all movies)

In [84]:
movies_merge = movies.merge(
    credits[['id', 'cast_top5', 'directors', 'writers']],
    on='id',
    how='left'
)

print("=== POST-MERGE VALIDATION ===")
print(f"Shape after merge: {movies_merge.shape}")
print(f"Rows: {len(movies_merge)} (should equal {len(movies)})")

# Check the missing movie
missing_credits = movies_merge[movies_merge['cast_top5'].isna()]
print(f"\nMovies missing credits after merge: {len(missing_credits)}")
if len(missing_credits) > 0:
    print("Sample of movies without credits:")
    print(missing_credits[['id', 'title', 'release_year', 'genres_list']].head())


# Verify Toy Story data transferred correctly
toy_story = movies_merge[movies_merge['id'] == 862]
print(f"\nToy Story verification:")
print(f"Title: {toy_story['title'].iloc[0]}")
print(f"Cast: {toy_story['cast_top5'].iloc[0]}")
print(f"Director: {toy_story['directors'].iloc[0]}")
print(f"Writer: {toy_story['writers'].iloc[0]}")

=== POST-MERGE VALIDATION ===
Shape after merge: (45424, 31)
Rows: 45424 (should equal 45424)

Movies missing credits after merge: 1
Sample of movies without credits:
           id         title  release_year genres_list
42845  401840  School's out          2017          []

Toy Story verification:
Title: Toy Story
Cast: ['Tom Hanks', 'Tim Allen', 'Don Rickles', 'Jim Varney', 'Wallace Shawn']
Director: ['John Lasseter']
Writer: ['Joss Whedon', 'Alec Sokolow', 'Joel Cohen', 'Andrew Stanton']


#### Verify types

In [86]:
print("\n=== DATA TYPE VERIFICATION ===")
print(f"cast_top5 type: {type(movies_merge['cast_top5'].iloc[0])}")
print(f"directors type: {type(movies_merge['directors'].iloc[0])}")
print(f"writers type: {type(movies_merge['writers'].iloc[0])}")
print(f"genres_list type: {type(movies_merge['genres_list'].iloc[0])}")


=== DATA TYPE VERIFICATION ===
cast_top5 type: <class 'list'>
directors type: <class 'list'>
writers type: <class 'list'>
genres_list type: <class 'list'>


In [134]:
imdb_movies = pd.read_csv("../data/raw/movie_metadata.csv")

In [135]:
imdb_movies.head(1)

,color,director_name,num_critic_for_reviews,duration,director_facebook_likes,actor_3_facebook_likes,actor_2_name,actor_1_facebook_likes,gross,genres,...,num_user_for_reviews,language,country,content_rating,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,3054.0,English,USA,PG-13,237000000.0,2009.0,936.0,7.9,1.78,33000


In [136]:
print("=" * 40)
print("IMDb METADATA - INITIAL INSPECTION")
print("=" * 40)

print(f"\nShape {imdb_movies.shape}")
print(f"\nFirst 5 columns: {imdb_movies.columns[:5].tolist()}")
print(f"\nData types:\n{imdb_movies.dtypes}")
print(f"\nSample of critical columns:")
critical_cols = ['movies_title', 'director_name', 'actor_1_name', 'plot_keywords', 'content_rating', 'imdb_score', 'movie_score', 'movie_imdb_link']

for col in critical_cols:
    if col in imdb_movies.columns:
        print(f"\n{col}:")
        print(f"    - Non-null: {imdb_movies[col].notna().sum()}/{len(imdb_movies)}")
        print(f"    - Sample: {imdb_movies[col].dropna().iloc[0] if imdb_movies[col].notna().any() else 'ALL NULL'}")

IMDb METADATA - INITIAL INSPECTION

Shape (5043, 28)

First 5 columns: ['color', 'director_name', 'num_critic_for_reviews', 'duration', 'director_facebook_likes']

Data types:
color                         object
director_name                 object
num_critic_for_reviews       float64
duration                     float64
director_facebook_likes      float64
actor_3_facebook_likes       float64
actor_2_name                  object
actor_1_facebook_likes       float64
gross                        float64
genres                        object
actor_1_name                  object
movie_title                   object
num_voted_users                int64
cast_total_facebook_likes      int64
actor_3_name                  object
facenumber_in_poster         float64
plot_keywords                 object
movie_imdb_link               object
num_user_for_reviews         float64
language                      object
country                       object
content_rating                object
budget    

#### Extract imdb_id from movie_imdb_link

In [137]:
import re

def extract_imdb_id(link):
    if pd.isna(link):
        return None
    match = re.search(r'(tt\d+)', str(link))
    return match.group(1) if match else None

imdb_movies['imdb_id'] = imdb_movies['movie_imdb_link'].apply(extract_imdb_id)
print(f"\nExtracted imdb_id - coverage: {imdb_movies['imdb_id'].notna().sum()}/{len(imdb_movies)}")


Extracted imdb_id - coverage: 5043/5043


#### Check if imdb_id exists in movies

In [138]:
if 'imdb_id' in movies.columns:
    print(f"imdb_id exists in movies. Coverage: {movies['imdb_id'].notna().sum()}/{len(movies)}")
    print(f"Sample: {movies['imdb_id'].dropna().iloc[:5].tolist()}")
else:
    print("imdb_id not in movies—need to extract from raw data")

imdb_id exists in movies. Coverage: 45454/45454
Sample: ['tt0114709', 'tt0113497', 'tt0113228', 'tt0114885', 'tt0113041']


#### Investigate the duplicates

In [139]:
duplicate_imdb_ids = imdb_movies[imdb_movies['imdb_id'].duplicated(keep=False)]
print(f"Total rows with duplicate IMDb IDs: {len(duplicate_imdb_ids)}")
print(f"Unique duplicate IMDb IDs: {duplicate_imdb_ids['imdb_id'].nunique()}")

Total rows with duplicate IMDb IDs: 241
Unique duplicate IMDb IDs: 117


In [140]:
example_dup = duplicate_imdb_ids['imdb_id'].iloc[0]
print(f"\nExample duplicate ID: {example_dup}")
dup_rows = duplicate_imdb_ids[duplicate_imdb_ids['imdb_id'] == example_dup]
for idx, row in dup_rows.iterrows():
    print(f"\nRow {idx}:")
    print(f"  Title: {row['movie_title']}")
    print(f"  Director: {row['director_name']}")
    print(f"  Year: {row['title_year']}")
    print(f"  Budget: {row['budget']}")


Example duplicate ID: tt0413300

Row 6:
  Title: Spider-Man 3 
  Director: Sam Raimi
  Year: 2007.0
  Budget: 258000000.0

Row 3461:
  Title: Spider-Man 3 
  Director: Sam Raimi
  Year: 2007.0
  Budget: 258000000.0


In [141]:
imdb_movies_dedup = imdb_movies.drop_duplicates(subset=['imdb_id'], keep='first')
print(f"\nIMDb after dedup: {len(imdb_movies_dedup)} rows (from {len(imdb_movies)})")


IMDb after dedup: 4919 rows (from 5043)


#### Merge with IMDb dataset on imdb_id

In [144]:
movies_final = movies_merge.merge(
    imdb_movies_dedup[['imdb_id', 'plot_keywords', 'content_rating', 
                       'director_name', 'actor_1_name', 'actor_2_name', 
                       'actor_3_name', 'imdb_score', 'gross', 
                       'movie_title', 'genres', 'budget', 'language']],
    left_on='imdb_id',
    right_on='imdb_id',
    how='left'
)

print("\n" + "=" * 60)
print("POST-FIX MERGE VALIDATION")
print("=" * 60)
print(f"Shape after fixed merge: {movies_final.shape}")
print(f"Rows: {len(movies_final)} (should equal original: {len(movies)})")

# Verify Toy Story now
toy_story = movies_final[movies_final['id'] == 862]
print("\n=== TOY STORY VERIFICATION ===")
print(f"IMDb ID: {toy_story['imdb_id'].iloc[0]}")
print(f"Plot Keywords: {toy_story['plot_keywords'].iloc[0]}")
print(f"Content Rating: {toy_story['content_rating'].iloc[0]}")
print(f"IMDb Score: {toy_story['imdb_score'].iloc[0]}")
print(f"Director (IMDb): {toy_story['director_name'].iloc[0]}")
print(f"Director (from credits): {toy_story['directors'].iloc[0]}")



# Check coverage
print("\n=== CRITICAL FIELD COVERAGE ===")
for field in ['plot_keywords', 'content_rating', 'directors', 'cast_top5']:
    if field in movies_final.columns:
        coverage = movies_final[field].notna().sum()
        pct = (coverage / len(movies_final)) * 100
        print(f"{field:20}: {coverage:6,}/{len(movies_final):,} ({pct:.2f}%)")


# Save the merged dataset
movies_final.to_csv('../data/processed/movies_merged.csv', index=False)
print("\n✓ Saved to ../data/processed/movies_merged.csv")


POST-FIX MERGE VALIDATION
Shape after fixed merge: (45424, 43)
Rows: 45424 (should equal original: 45454)

=== TOY STORY VERIFICATION ===
IMDb ID: tt0114709
Plot Keywords: claw crane|cowboy|jealousy|rivalry|toy
Content Rating: G
IMDb Score: 8.3
Director (IMDb): John Lasseter
Director (from credits): ['John Lasseter']

=== CRITICAL FIELD COVERAGE ===
plot_keywords       :  4,526/45,424 (9.96%)
content_rating      :  4,435/45,424 (9.76%)
directors           : 45,423/45,424 (100.00%)
cast_top5           : 45,423/45,424 (100.00%)

✓ Saved to ../data/processed/movies_merged.csv


#### Load Wikipedia plots JSON

In [145]:
import json

with open('../data/external/wikipedia_plots.json', 'r') as f:
    wiki_plots = json.load(f)

print(f"Wikipedia plots loaded: {len(wiki_plots)} movies")
print(f"Sample TMDB ID: {list(wiki_plots.keys())[0]}")
print(f"Sample plot (first 200 chars): {list(wiki_plots.values())[0][:200]}...")

# Convert to DataFrame for merging
wiki_df = pd.DataFrame([
    {'tmdb_id': int(tmdb_id), 'wiki_plot': plot}
    for tmdb_id, plot in wiki_plots.items()
])

print(f"\nWiki DataFrame shape: {wiki_df.shape}")
print(f"Unique TMDB IDs in wiki: {wiki_df['tmdb_id'].nunique()}")

# Merge with movies_final
movies_final = movies_final.merge(
    wiki_df,
    left_on='id',
    right_on='tmdb_id',
    how='left'
)

print(f"\nAfter wiki merge: {movies_final.shape}")
print(f"Movies with Wikipedia plots: {movies_final['wiki_plot'].notna().sum()}/{len(movies_final)}")
print(f"Coverage: {movies_final['wiki_plot'].notna().mean()*100:.2f}%")

Wikipedia plots loaded: 5031 movies
Sample TMDB ID: 27205
Sample plot (first 200 chars): Inception is a 2010 science fiction action film written and directed by Christopher Nolan, who also produced it with Emma Thomas, his wife. The film stars Leonardo DiCaprio as a professional thief who...

Wiki DataFrame shape: (5031, 2)
Unique TMDB IDs in wiki: 5031

After wiki merge: (45424, 45)
Movies with Wikipedia plots: 5031/45424
Coverage: 11.08%
